# ⚙️ 개인 설정 UI
이름, 생활비, 위험 성향, 목표 자산 비중을 화면에서 입력하고 **config.yaml**에 저장합니다.

In [ ]:
import yaml
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pathlib import Path

CONFIG_PATH = Path('config.yaml')

# ── 기존 설정 로드 ──────────────────────────────────────────
def load_config():
    if CONFIG_PATH.exists():
        with open(CONFIG_PATH, encoding='utf-8') as f:
            return yaml.safe_load(f) or {}
    return {}

cfg = load_config()
alloc = cfg.get('target_allocation', {})

# monthly_expense: user 하위 또는 최상위 어디서든 읽기
def get_expense(cfg):
    return cfg.get('user', {}).get('monthly_expense') or cfg.get('monthly_expense', 2500000)

display(HTML('<h2 style="color:#2c3e50;margin-bottom:4px">⚙️ 개인 설정</h2><hr style="margin-top:0">'))

In [ ]:
# ── 위젯 정의 ───────────────────────────────────────────────
style   = {'description_width': '160px'}
layout  = widgets.Layout(width='420px')

w_name = widgets.Text(
    value=cfg.get('user', {}).get('name', ''),
    description='이름',
    style=style, layout=layout
)
w_birth = widgets.IntText(
    value=cfg.get('user', {}).get('birth_year', 1965),
    description='출생연도',
    style=style, layout=layout
)
w_expense = widgets.IntText(
    value=get_expense(cfg),
    description='월 생활비 (원)',
    style=style, layout=layout
)
w_risk = widgets.Dropdown(
    options=['conservative', 'balanced', 'aggressive'],
    value=cfg.get('risk_profile', 'balanced'),
    description='위험 성향',
    style=style, layout=layout
)
w_rebal = widgets.IntSlider(
    value=int(cfg.get('rebalance_threshold', 10)),
    min=5, max=25, step=1,
    description='리밸런싱 기준 (%)',
    style=style, layout=widgets.Layout(width='500px')
)

display(HTML('<h3 style="color:#34495e">👤 기본 정보</h3>'))
display(w_name, w_birth, w_expense, w_risk, w_rebal)

In [ ]:
# ── 목표 비중 위젯 ──────────────────────────────────────────
display(HTML('<h3 style="color:#34495e;margin-top:16px">🎯 목표 자산 비중</h3>'))
display(HTML('<p style="color:#7f8c8d;font-size:13px">합계가 100%가 되도록 조정하세요.</p>'))

alloc_style  = {'description_width': '100px'}
alloc_layout = widgets.Layout(width='380px')

w_cash   = widgets.IntSlider(value=int(alloc.get('cash',   25)), min=0, max=100, step=1,
                              description='현금 (%)',   style=alloc_style, layout=alloc_layout)
w_bond   = widgets.IntSlider(value=int(alloc.get('bond',   25)), min=0, max=100, step=1,
                              description='채권 (%)',   style=alloc_style, layout=alloc_layout)
w_equity = widgets.IntSlider(value=int(alloc.get('equity', 35)), min=0, max=100, step=1,
                              description='주식 (%)',   style=alloc_style, layout=alloc_layout)
w_income = widgets.IntSlider(value=int(alloc.get('income', 15)), min=0, max=100, step=1,
                              description='배당·리츠 (%)', style=alloc_style, layout=alloc_layout)

total_label = widgets.HTML()

def update_total(*args):
    total = w_cash.value + w_bond.value + w_equity.value + w_income.value
    color = '#27ae60' if total == 100 else '#e74c3c'
    total_label.value = (
        f'<div style="margin-top:8px;font-size:15px;color:{color};font-weight:bold">'
        f'합계: {total}% {"✅" if total==100 else "⚠️ (100%가 되어야 합니다)"}'
        f'</div>'
    )

for w in [w_cash, w_bond, w_equity, w_income]:
    w.observe(update_total, names='value')

update_total()
display(w_cash, w_bond, w_equity, w_income, total_label)

In [ ]:
# ── 저장 버튼 ───────────────────────────────────────────────
btn_save   = widgets.Button(description='💾 저장', button_style='success',
                             layout=widgets.Layout(width='150px', height='40px'))
btn_reload = widgets.Button(description='🔄 불러오기', button_style='info',
                             layout=widgets.Layout(width='150px', height='40px'))
out = widgets.Output()

def on_save(b):
    with out:
        clear_output()
        total = w_cash.value + w_bond.value + w_equity.value + w_income.value
        if total != 100:
            display(HTML(f'<div style="color:#e74c3c;font-size:14px">'
                         f'❌ 목표 비중 합계가 {total}%입니다. 100%로 맞춰주세요.</div>'))
            return

        expense = w_expense.value

        new_cfg = {
            'user': {
                'name':            w_name.value.strip(),
                'birth_year':      w_birth.value,
                'monthly_expense': expense   # ← user 하위에도 저장
            },
            'monthly_expense':     expense,  # ← 최상위에도 저장 (하위 호환)
            'risk_profile':        w_risk.value,
            'rebalance_threshold': w_rebal.value,
            'target_allocation': {
                'cash':   w_cash.value,
                'bond':   w_bond.value,
                'equity': w_equity.value,
                'income': w_income.value
            }
        }

        # 기존 항목 유지 (portfolio, withdrawal, alert 등)
        existing = load_config()
        for k, v in new_cfg.items():
            existing[k] = v

        with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
            yaml.dump(existing, f, allow_unicode=True, default_flow_style=False, sort_keys=False)

        display(HTML('<div style="color:#27ae60;font-size:14px;font-weight:bold">'
                     '✅ config.yaml 저장 완료!</div>'))

def on_reload(b):
    with out:
        clear_output()
        display(HTML('<div style="color:#2980b9;font-size:13px">'
                     '🔄 현재 저장된 값을 다시 불러오려면 커널을 재시작 후 실행하세요.</div>'))

btn_save.on_click(on_save)
btn_reload.on_click(on_reload)

display(HTML('<br>'))
display(widgets.HBox([btn_save, btn_reload]))
display(out)